# Génération d’images avec Python, DALL-E et StableDiffusion

Lino Galiana  
2023-12-20

Pour tester les exemples présentés dans ce chapitre:

<div class="alert alert-info" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #007bff80;">
<h3 class="alert-heading"><i class="fa fa-comment"></i> Note</h3>

L’utilisation de ce tutoriel est assez exigeante en termes d’infrastructure
car il est nécessaire de disposer de **GPU**.
Les GPU sont des **ressources rares et assez chères**, elle ne sont donc pas mises à disposition de façon
aussi aisées que les CPU dans les *cloud providers*. Il s’agit de plus
de ressources **plus polluantes** que les CPU.

Des GPU sont disponibles sur `Google Colab`, la procédure pour les activer
est expliquée ci-dessous. Des GPU sont également disponibles sur le `SSPCloud`
mais sont à utiliser avec parcimonie. Elles ne sont pas mises à disposition
par défaut car il s’agit d’une ressource rare. Ce chapitre, lancé depuis le
bouton en début de chapitre, active cette option pour permettre la réplication
des exemples.

</div>

<div class="alert alert-warning" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #ffc10780;">
<h3 class="alert-heading"><i class="fa fa-lightbulb-o"></i> Hint</h3>

Par défaut, `Colab` n’utilise pas de GPU mais de la CPU. Il est donc nécessaire
d’éditer les paramètres d’exécution du Notebook

-   Dans le menu `Exécution`, cliquer sur `Modifier le type d'exécution` ;
-   Sélectionner `GPU` sous `Accélérateur matériel`.

</div>

# Contexte

La publication en avril 2022 par l’organisation [Open AI](https://openai.com/) de
son modèle de génération de contenu créatif [`Dall-E-2`](https://openai.com/dall-e-2/)
(un jeu de mot mélangeant Dali et Wall-E) a créé un bruit inédit dans
le monde de la *data science*.
A propos de `Dall-E`, le bloggeur *tech* Casey Newton a pu parler d’une
[révolution créative dans le monde de l’IA](https://www.platformer.news/p/how-dall-e-could-power-a-creative).
*[The Economist](https://www.economist.com/news/2022/06/11/how-a-computer-designed-this-weeks-cover)* a par consacré
un numéro au sujet de l’émergence de ces intelligences artificielles
créatrices de contenu.

Ce bruit sur la capacité des
intelligences artificielle à générer du contenu créatif
a d’ailleurs été amplifié plus récemment
avec la publication du *chatbot* `chatGPT`
(voir [cet éditorial du *Guardian*](https://www.theguardian.com/commentisfree/2022/dec/10/i-wrote-this-column-myself-but-how-long-before-a-chatbot-could-do-it-for-me)).

L’inconvénient principal de `Dall-E`
pour générer facilement du contenu
est que le nombre de contenu pouvant être créé
avec un accès gratuit est limité (50 crédits gratuits par mois).
Depuis le 22 Août 2022, un générateur de contenu
similaire est disponible gratuitement,
avec une licence plus permissive<a name="cite_ref-3"></a>[<sup>\[3\]</sup>](#cite_note-3). Ce générateur, développé
par une équipe de chercheurs (Rombach et al. 2022),
s’appelle `Stable Diffusion` ([dépôt `Github` pour le code source](https://github.com/CompVis/stable-diffusion) et
[dépôt `HuggingFace` pour le modèle mis à disposition](https://huggingface.co/CompVis/stable-diffusion-v1-4)<a name="cite_ref-4"></a>[<sup>\[4\]</sup>](#cite_note-4)).
Un [excellent article de blog](https://huggingface.co/blog/stable_diffusion) décrit la démarche de `Stable Diffusion`. La plupart des exemples originaux
dans cette partie seront basés sur `Stable Diffusion`.

Il est notamment possible de réutiliser l’image générée à des fins commerciales. En revanche, il est interdit de chercher à nuire à une personne. Pour cette raison, il est fréquent que les visages de personnes célèbres soient floutés pour éviter la création de contenu nuisant à leur réputation.

<div class="alert alert-info" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #007bff80;">
<h3 class="alert-heading"><i class="fa fa-comment"></i> <code>HuggingFace</code></h3>

`Huggingface` est une plateforme de partage de modèles de type réseau de neurone. Les utilisateurs de réseaux de neurones peuvent
ainsi mettre à disposition le résultat de leurs travaux sous forme d’API pour faciliter la réutilisation de leurs
modèles ou réutiliser facilement des modèles, ce qui évite de les ré-entraîner (ce qui aurait un coût écologique non
négligeable comme expliqué dans le chapitre introductif).

</div>

`Dall-E-2` et `StableDiffusion`
sont des modèles généralistes.
D’autres modèles, plus spécialisés,
existent également.
Le modèle [`Midjourney`](https://en.wikipedia.org/wiki/Midjourney)
(produit propriétaire de la société du même nom)
permet la production de contenu
artistique, [DreamBooth](https://dreambooth.github.io/) (développé par Google)
est spécialisé dans la génération de contenu dans un nouveau
contexte.

Le principe de tous ces modèles est le même: un utilisateur
donne une instruction (une ou plusieurs phrases) et l’intelligence
artificielle l’interprète et génère une image censée être
cohérente avec l’instruction.

Voici par exemple l’une des productions possibles de `DALL-E-2`

![](https://upload.wikimedia.org/wikipedia/commons/2/2b/A_Shiba_Inu_dog_wearing_a_beret_and_black_turtleneck_DALLE2.jpg)

*“A Shiba Inu dog wearing a beret and black turtleneck”*

`Midjourney`, spécialisé dans le contenu esthétique,
génèrera l’image suivante avec l’instruction *“mechanical dove”* :

![](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a9/A_mechanical_dove_8274822e-52fe-40fa-ac4d-f3cde5a332ae.png/250px-A_mechanical_dove_8274822e-52fe-40fa-ac4d-f3cde5a332ae.png)

`StableDiffusion`, modèle généraliste comme `Dall-E`,
crééra le contenu suivant avec
l’instruction *“A photograph of an astronaut riding a horse”* :

![“A photograph of an astronaut riding a horse”](https://upload.wikimedia.org/wikipedia/commons/thumb/3/32/A_photograph_of_an_astronaut_riding_a_horse_2022-08-28.png/250px-A_photograph_of_an_astronaut_riding_a_horse_2022-08-28.png)

Enfin, `DreamBooth` pourra lui introduire un chien dans une grande variété
de contextes:

![](https://dreambooth.github.io/DreamBooth_files/teaser_static.jpg)

Un compte *Twitter* ([Weird AI Generations](https://twitter.com/weirddalle))
propose de nombreuses générations de contenu drôles ou incongrues.
Voici un premier exemple de production humoristique faite à partir de Mini Dall-E, la version
publique:

<blockquote class="twitter-tweet"><p lang="zxx" dir="ltr"><a href="https://t.co/DIerJPtXGE">pic.twitter.com/DIerJPtXGE</a></p>&mdash; Weird Ai Generations (@weirddalle) <a href="https://twitter.com/weirddalle/status/1556027692163760130?ref_src=twsrc%5Etfw">August 6, 2022</a></blockquote> <script async src="https://platform.twitter.com/widgets.js" charset="utf-8"></script>

Ainsi qu’un deuxième:

<blockquote class="twitter-tweet"><p lang="zxx" dir="ltr"><a href="https://t.co/Ju0Pdcokth">pic.twitter.com/Ju0Pdcokth</a></p>&mdash; Weird Ai Generations (@weirddalle) <a href="https://twitter.com/weirddalle/status/1556573904600268801?ref_src=twsrc%5Etfw">August 8, 2022</a></blockquote> <script async src="https://platform.twitter.com/widgets.js" charset="utf-8"></script>

Les modèles `Dall-E-2` et `Stable Diffusion`
s’appuient sur des réseaux de neurones à différents niveaux :

-   le contenu de la phrase est analysé par un réseau de neurones similaire (mais bien sûr plus évolué) que
    ceux que nous avons présenté dans la partie [NLP](#nlp)
-   les éléments importants de la phrase (recontextualisés) sont ensuite transformés en image à partir de
    modèles entraînés à reconnaître des images

![](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/stable_diffusion.png)

Illustration du fonctionnement de ce type de générateur d’image (ici à partir de `Stable Diffusion`)

# Générer du contenu avec Dall-E via l’API d’OpenAI

<div class="alert alert-danger" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left: .2rem solid #ff0039;">
<i class="fa fa-exclamation-triangle"></i> Warning</h3>

Les services d’`OpenAI` ne sont gratuits que dans une certaine
limite. Votre clé d’API est donc assez précieuse car si elle
est usurpée, elle peut permettre à certaines personnes
d’épuiser vos crédits gratuits voire d’utiliser des crédits
payants à votre place.

Si vous êtes enregistrés récemment dans le service d’API
d’`OpenAI`, vous avez accès à des crédits gratuits. Ne les
utilisez néanmoins pas avec trop de légèreté en ne contrôlant
pas les paramètres de vos appels aux API car ces crédits
sont pour l’ensemble des services d’`OpenAI`(`chatGPT`,
`Dall-E`, `DaVinci`…)

</div>

Le contenu de cette partie s’appuie sur
le tutoriel du site [realpython](https://realpython.com/generate-images-with-dalle-openai-api/)
L’utilisation de `Dall-E` sera faite via le package `openai` qui donne
accès à l’[API d’OpenAI](https://beta.openai.com/docs/api-reference?lang=python).
Pour l’installer depuis la cellule d’un `Notebook`:

In [1]:
!pip install openai

Après avoir obtenu votre [clé d’API](https://realpython.com/generate-images-with-dalle-openai-api/#get-your-openai-api-key), on va supposer que celle-ci
est stockée dans une variable `key`:

In [2]:
key = "sk-XXXXXXXXXX" #remplacer avec votre clé

Ensuite, l’utilisation de l’API est assez directe:

In [3]:
openai.api_key = key
openai.Image.create(
  prompt="Teddy bears working on new AI research underwater with 1990s technology",
  n=2,
  size="1024x1024"
)

L’*output* est un JSON avec les URL des images générées.
Voici les deux images générées :

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/dallE.png)

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/dallE2.png)

Pour aller plus loin, vous pouvez consulter
le [tutoriel de `realpython`](https://realpython.com/generate-images-with-dalle-openai-api/#get-your-openai-api-key)

# Comment utiliser `Stable Diffusion` ?

[`Stable Diffusion`](https://github.com/CompVis/stable-diffusion) est
une intelligence artificielle créatrice de contenu qui permet de
générer du contenu à partir d’une phrase - ce pour quoi nous allons
l’utiliser - mais aussi modifier des images à partir d’instructions.

`Stable Diffusion` est un modèle plus pratique à utiliser depuis `Python`
que `Dall-E`. Celui-ci
est *open-source* et peut être téléchargé et réutilisé directement depuis `Python`.
La méthode la plus pratique est d’utiliser le modèle mis
à disposition sur `HuggingFace`. Le modèle est implémenté
à travers le *framework* [`PyTorch`](https://pytorch.org/).

[`PyTorch`](https://pytorch.org/), librairie développée
par `Meta`, n’est pas implementé directement en `Python`
pour des raisons de performance mais en `C++` - `Python` étant un
langage lent, le revers de la médaille de sa facilité
d’usage. A travers `Python`, on va utiliser une API haut niveau
qui va contrôler la structure des réseaux de neurones ou
créer une interface entre des
données (sous forme d’array `Numpy`) et le modèle.
Pour ce type de *packages* qui utilisent un langage compilé,
l’installation via `Pandas`

<details><summary>Configuration spécifique à <code>Colab</code> 👇</summary>

Sur `Colab`, `conda` n’est pas disponible par défaut.
Pour pouvoir
installer un package en utilisant `conda` sur `Colab`,
on utilise donc l’astuce
suivante :

``` python
!pip install -q condacolab
import condacolab
condacolab.install()
```

</details>

## Installation de `PyTorch`

Pour installer `PyTorch`, la librairie de *Deep Learning*
développée par `Meta`, il suffit de suivre les recommandations
sur le [site web officiel](https://pytorch.org/).
Dans un `Notebook`, cela prendra la forme suivante :

In [4]:
!conda install mamba
!mamba install pytorch torchvision torchaudio cudatoolkit=10.2 -c pytorch

<div class="alert alert-info" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #007bff80;">
<h3 class="alert-heading"><i class="fa fa-comment"></i> Note</h3>

Je propose ici d’utiliser `mamba` pour accélérer l’installation.
Des éléments sur `mamba` sont disponibles dans l’introduction de ce cours.

</div>

## Accès à `HuggingFace`

La question - non négligeable - de l’accès à
de la GPU mise à part,
la réutilisation des modèles de `Stable Diffusion` est
très facile car la documentation mise à disposition sur
`HuggingFace` est très bien faite.

La première étape est de se créer un compte sur `HuggingFace`
et se créer un *token*<a name="cite_ref-4"></a>[<sup>\[4\]</sup>](#cite_note-4). Ce *token* sera donné à l’API
de `HuggingFace` pour s’authentifier.

Comme les autres plateformes du monde de la *data science*,
`HuggingFace` a adopté l’utilisation standardisée des
jetons (*token*) comme méthode d’authentification. Le jeton est
comme un mot de passe sauf qu’il n’est pas inventé par l’utilisateur
(ce qui permet qu’il ne soit pas partagé avec d’autres sites web potentiellement
moins sécurisés), est révocable (date d’expiration ou choix de l’utilisateur)
et dispose de droits moins importants qu’un
mot de passe qui vous permet, potentiellement,
de changer tous les paramètres de votre compte. Je recommande vivement l’utilisation
d’un gestionnaire de mot de passe pour
stocker vos *token* (si vous utilisez `Git`, `Docker`, etc.
vous en avez potentiellement beaucoup) plutôt que
stocker ces jetons dans des fichiers non sécurisés.

L’API d’`HuggingFace` nécessite l’installation du
package [`diffusers`](https://huggingface.co/docs/transformers/installation).
Dans un `Notebook`, le code suivant permet d’installer la librairie
requise:

In [5]:
!pip install --upgrade diffusers transformers scipy accelerate

<div class="alert alert-info" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #007bff80;">
<h3 class="alert-heading"><i class="fa fa-comment"></i> Note</h3>

On va supposer que le *token* est stocké dans une variable
d’environnement `HF_PAT`. Cela évite d’écrire le *token*
dans un *Notebook* qu’on va
potentiellement partager, alors que le *token*
est un élément à garder secret. Pour l’importer
dans la session `Python`:

Si vous n’avez pas la possibilité de rentrer le token dans les variables
d’environnement, créez une cellule qui crée la variable
`HF_TOKEN` et supprimez là de suite pour ne pas l’oublier avant
de partager votre token.

</div>

In [6]:
import os
HF_TOKEN = os.getenv('HF_PAT')

# Générer une image

On va créer l’image suivante :

> **“Chuck Norris fighting against Zeus on Mount Olympus in an epic Mortal Kombat scene”**

Pas mal comme scénario, non ?!

<div class="alert alert-info" role="alert" style="color: rgba(0,0,0,.8); background-color: white; margin-top: 1em; margin-bottom: 1em; margin:1.5625emauto; padding:0 .6rem .8rem!important;overflow:hidden; page-break-inside:avoid; border-radius:.25rem; box-shadow:0 .2rem .5rem rgba(0,0,0,.05),0 0 .05rem rgba(0,0,0,.1); transition:color .25s,background-color .25s,border-color .25s ; border-right: 1px solid #dee2e6 ; border-top: 1px solid #dee2e6 ; border-bottom: 1px solid #dee2e6 ; border-left:.2rem solid #007bff80;">
<h3 class="alert-heading"><i class="fa fa-comment"></i> Note</h3>

Pour que les résultats soient reproductibles entre différentes
sessions,
nous allons fixer
la racine du générateur aléatoire.

</div>

In [7]:
import torch
generator = torch.Generator("cuda").manual_seed(123)

Si vous voulez vous amuser à explorer différents résultats
pour un même texte, vous pouvez ne pas fixer de racine aléatoire.
Dans ce cas, retirer l’argument `generator` des codes présentés
ultérieurement.

Nous allons donc utiliser l’instruction suivante :

In [8]:
prompt = "Chuck Norris fighting against Zeus on Mount Olympus in an epic Mortal Kombat scene"

L’initialisation du modèle se fait de la manière
suivante :

In [9]:
import torch
from torch import autocast
from diffusers import StableDiffusionPipeline

model_id = "CompVis/stable-diffusion-v1-4"
device = "cuda"

generator = torch.Generator("cuda").manual_seed(1024)

Enfin, pour générer l’image:

In [10]:
pipe = StableDiffusionPipeline.from_pretrained(model_id, use_auth_token=HF_TOKEN, generator=generator)
pipe = pipe.to(device)

with autocast("cuda"):
    image = pipe(prompt, guidance_scale=7.5, generator = generator)["images"][0]  

   
image.save("featured.png")

Qui peut être visualisé avec le code suivant, dans un `notebook`:

In [11]:
from IPython.display import Image 
pil_img = Image(filename="featured.png")
display(pil_img)

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/featured.png)

C’est une représentation assez fidèle du
pitch *“Chuck Norris fighting against Zeus on Mount Olympus in an epic Mortal Kombat scene”* :boom:.
Y a un petit côté [*Les Dix Commandements*](https://fr.wikipedia.org/wiki/Les_Dix_Commandements_(film,_1956)#/media/Fichier:Charlton_Heston_in_The_Ten_Commandments_film_trailer.jpg) que j’aime bien.

En voici une autre que j’aime bien (mais malheureusement je ne peux la reproduire car je n’ai pas
gardé en mémoire la racine l’ayant généré :sob:)

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/chuck.png)

Il est également possible de générer plusieurs images du même texte (voir
la [note de blog](https://huggingface.co/blog/stable_diffusion) de l’équipe
à l’origine de `Stable Diffusion`). Cependant, c’est assez exigeant en
mémoire et cela risque d’être impossible sur `Colab` (y compris
en réduisant le poids des vecteurs numériques comme proposé dans le *post*)

# Bonus

Pour le plaisir, voici `PuppyMan`, le dernier né du Marvel Universe:

![](https://minio.lab.sspcloud.fr/lgaliana/generative-art/pythonds/puppyman.png)

In [12]:
prompt = "In a new Marvel film we discover puppyman a new super hero that is half man half bulldog"

In [13]:
import torch
from torch import autocast
from diffusers import StableDiffusionPipeline

model_id = "CompVis/stable-diffusion-v1-4"
device = "cuda"

generator = torch.Generator("cuda").manual_seed(1024)

pipe = StableDiffusionPipeline.from_pretrained(model_id, use_auth_token=HF_TOKEN, generator=generator)
pipe = pipe.to(device)

with autocast("cuda"):
    image = pipe(prompt, guidance_scale=7.5, generator = generator)["images"][0]  

   
image.save("puppyman.png")

La moitié humain semble être son costume de super-héros, pas la bipédie.
Mais le rendu
est quand même épatant !

A vous de jouer :hugging_face:

# Références

Rombach, Robin, Andreas Blattmann, Dominik Lorenz, Patrick Esser, and Björn Ommer. 2022. “High-Resolution Image Synthesis with Latent Diffusion Models.” In *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*, 10684–95.